# AnyCam — 04 · Step 6: does the hyperspectral model work on other cameras? (E1 + E2)

1. Re-train our Step 5 baseline (model A: ResNet18, 2x images, strong augmentation, all 204 bands) — ~25 min
2. Train three **camera-specific** models (drone MicaSense, Sentinel-2 satellite, phone RGB) — the "one model per camera" approach used today — ~25 min each
3. Test the 204-band model on **7 simulated cameras**, each: ideal / band drift / calibration+noise / both — ~15 min

Total ~2 h. Same setup: **Add Input → HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**.
Best: **Save Version → Save & Run All**, then save to GitHub as `kaggle_runs/04_step6_camera_test.ipynb`.

In [ ]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

In [ ]:
# 1) the 204-band baseline (model A from Step 5)
!python experiments/train_baseline.py --data {DATA} --split official --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/A_hs204

In [ ]:
# 2) camera-specific models: trained and tested on the same simulated camera
for cam in ['micasense_rededge_mx', 'sentinel2a_vnir', 'generic_rgb']:
    !python experiments/train_baseline.py --data {DATA} --split official --camera {cam} --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/oracle_{cam}

In [ ]:
# 3) test the 204-band model on all 7 cameras (+ calibration errors) and draw the figure
!python experiments/eval_cameras.py --data {DATA} --run runs/A_hs204 --oracles "runs/oracle_*" --out runs/step6_eval
from IPython.display import Image, display
display(Image('runs/step6_eval/camera_eval.png'))

## Done
When every cell has finished: **GitHub → file** `kaggle_runs/04_step6_camera_test.ipynb` → **Save**. Tell Claude "done".